In [ ]:
from tensorflow.keras.layers import Input, Dense, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.models import Sequential
model = Sequential([
                    Input((224,224,3)),
                    Conv2D(16,(3,3), activation='relu'),
                    MaxPooling2D((2,2)),
                    Conv2D(32,(3,3), activation='relu'),
                    MaxPooling2D((2,2)),
                    Flatten(),
                    Dense(32, activation='relu'),
                    Dense(64, activation='sigmoid'),
                    Dense(1, activation='sigmoid')
                  ])

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
train_generator = ImageDataGenerator(rescale=1.0/255)

In [ ]:
train_data = train_generator.flow_from_directory(r'E:\Luminar\ML\05. Deep Learning\02. Dog_Cat_Classification\cat-and-dog', target_size=(224,224), class_mode='binary', shuffle=False)

In [ ]:
model.fit(train_data, epochs=10)

In [ ]:
test_data = train_generator.flow_from_directory(r'E:\Luminar\ML\05. Deep Learning\02. Dog_Cat_Classification\cat-and-dog\test_set', target_size=(224,224), class_mode='binary')

In [ ]:
model.evaluate(test_data)

In [ ]:
y_pred = model.predict(test_data)>0.5

In [ ]:
y_test = test_data.classes

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, roc_curve, confusion_matrix, auc


print(accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred))
sns.heatmap(confusion_matrix(y_test,y_pred), annot=True, fmt='d')
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

fpr, tpr, thresholds = roc_curve(y_test, y_pred)
roc_auc = auc(fpr, tpr)
plt.figure()
plt.plot(fpr, tpr,
label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
import cv2

In [ ]:
img_array = cv2.imread('/content/WhatsApp Image 2025-08-28 at 11.52.36_2b4b228b.jpg')

In [ ]:
img_array.shape

In [ ]:
img_res = cv2.resize(img_array,(224,224))
img_res = img_res.reshape((1,224,224,3))

In [ ]:
img_res = img_res/255
model.predict(img_res)

In [ ]:
train_data.class_indices

In [ ]:
def predict(path):
  img_arr=cv2.imread(path)
  img_res=cv2.resize(img_arr,(224,224))
  img_reshape=img_res.reshape((1,224,224,3))/255
  res=model.predict(img_reshape)[0][0]
  if res>0.5:
    print("Dog")
  else:
    print("cat")

Transfer Learning

In [ ]:
from tensorflow.keras.applications.vgg16 import  VGG16
base_model = VGG16(include_top=False, input_shape=(224,224,3), weights='imagenet')

In [ ]:
base_model.summary()

In [ ]:
for layer in base_model.layers:
  layer.trainable=False

In [ ]:
final_model = Sequential([base_model, Flatten(),
                          Dense(500, activation='relu'),
                          Dense(1, activation='sigmoid')])

In [ ]:
final_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
final_model.fit(train_data, epochs=10)

In [ ]:
final_model.evaluate(test_data)

In [ ]:
final_model.save('Final_model.keras')

In [ ]:
from tensorflow.keras.models import load_model
loaded_model = load_model('Final_model.keras')
loaded_model.summary()

In [ ]:
raw_pred = loaded_model.predict(img_res)
raw_pred

In [ ]:
label = predict('/content/WhatsApp Image 2025-08-28 at 11.52.36_2b4b228b.jpg')